# $\nu_e$ CC Inclusive — Cross-Section Extraction (Wiener-SVD)

Takes the selection (`nuecc_dfs/selected_nuecc_qual.df`) and the covariances written by
**`nue_syst.ipynb` §2.11** (`saved_syst/<var>/<family>_cov_matrices.npz`, plus `bkg_extra_cv.npz` from §2.11b)
and extracts the flux-averaged total and differential cross sections in reco-$e^-$ KE and $\cos\theta$.
Plots follow L. Tung's SBND $\nu_e$CC technote (figure numbers quoted in the headings).

| § | Content | Technote |
|---|---|---|
| 0 | Config | |
| 1 | Load selection, signal-definition checks | §2.1 |
| 2 | Response matrix $r_{a\mu}=\epsilon_\mu\Delta_{a\mu}$, migration, efficiency | Eq. 32, Fig. 33 |
| 3 | Background, covariances, expected event rate, uncertainty budget | Figs. 5, 19–22 |
| 4 | Wiener-SVD: closure, nominal Asimov unfolding, $A_C$, unfolded covariance, regularization | Figs. 34, 40–42, 63–68 |
| 5 | Fake-data tests (unconstrained) | §5.3.2, Figs. 35–38 |
| 6 | Total + differential cross section | Eq. 29–30, Table 7, Fig. 39 |
| 7 | Wiener-SVD vs OmniFold | |
| 8 | Sanity checks & summary | |

**Run order:** `nue_selection.ipynb` → `nue_syst.ipynb` (incl. §2.11b) → this notebook (*Kernel → Restart & Run All*).
Everything here is **Asimov / MC** — no on-beam data is unfolded.

## 0. Config

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os, sys, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from IPython.display import display

CAFPYANA_WD = '/home/castalyf/cafpyana'
for p in [CAFPYANA_WD, CAFPYANA_WD + '/pyanalib']:
    if p not in sys.path:
        sys.path.insert(0, p)

from analysis_village.unfolding.wienersvd import WienerSVD
import nue_helpers as nh
import nue_selection as nue_sel
import nue_xsec_helpers as xh
from nue_xsec_helpers import VariableConfig, step, plot_matrix

mpl.rcParams.update({
    'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 9,
    'figure.dpi': 110, 'axes.grid': False,
})
print('Imports OK')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DF_OUT_DIR   = './nuecc_dfs'
SYST_DIR     = './saved_syst'                 # written by nue_syst.ipynb §2.11
PLOT_DIR     = './plots_xsec'
COV_OUT_DIR  = './nuecc_cov'
OMNIFOLD_COV = '/home/castalyf/Omnifold_SBND/sbnd/covariance/'
SEL_FILE     = f'{DF_OUT_DIR}/selected_nuecc_qual.df'

# ── Exposure ──────────────────────────────────────────────────────────────────
TARGET_POT   = 6.6e20      # POT the saved covariances / CVs are normalized to (nue_syst)
EXPOSURE_POT = TARGET_POT  # POT of the expected (Asimov) measurement; e.g. 8.81e19 = technote exposure.
                           # CVs scale ∝ POT, syst. covariances ∝ POT², data stat. = Poisson at this POT.

# ── Analysis choices ──────────────────────────────────────────────────────────
FINAL_STAGE = f"sel_{nue_sel.CUT_NAMES_MORE[-1]}"     # same definition as nue_syst.ipynb
C_TYPE, NORM_TYPE = 2, 0      # Wiener-SVD assisting matrix (C2 = curvature) and model-norm exponent
TITLE     = r'SBND $\nu_e$ CC Inclusive'
POT_LABEL = rf'{EXPOSURE_POT:.2e} POT (Asimov)'
N_TARGETS, FLUX_PER_POT = xh.get_constants()
PHI = FLUX_PER_POT * EXPOSURE_POT                     # integrated νe flux [cm⁻²]

SAVE_FIGS = True
for d in [COV_OUT_DIR] + [f'{PLOT_DIR}/{s}' for s in
          ('', 'response', 'uncertainty', 'unfolding', 'fakedata', 'xsec', 'omnifold_comparison')]:
    os.makedirs(d, exist_ok=True)

def savefig(fig, name, sub=''):
    """Save to plots_xsec/<sub>/<name>.png, show inline, close."""
    if SAVE_FIGS:
        fig.savefig(os.path.join(PLOT_DIR, sub, f'{name}.png'), dpi=150, bbox_inches='tight')
    display(fig)
    plt.close(fig)

print(f'FINAL_STAGE = {FINAL_STAGE}')
print(f'N_targets   = {N_TARGETS:.4e} nucleons')
print(f'Φ           = {FLUX_PER_POT:.3e} cm⁻²/POT × {EXPOSURE_POT:.2e} POT = {PHI:.3e} cm⁻²')

In [ ]:
# Binning: xh.ANALYSIS_BINS must equal nue_syst.ipynb §0.2 (checked against the saved CVs in §3)
VAR_CFGS = {
    'reco_ke': VariableConfig(
        'reco_ke', xh.ANALYSIS_BINS['reco_ke'], r'Reco leading-$e^-$ KE [MeV]',
        xlabel_true=r'True leading-$e^-$ KE [MeV]', disp_widths=xh.ANALYSIS_WIDTHS['reco_ke']),
    'reco_costheta': VariableConfig(
        'reco_costheta', xh.ANALYSIS_BINS['reco_costheta'], r'Reco leading-$e^-$ $\cos\theta$',
        xlabel_true=r'True leading-$e^-$ $\cos\theta$', disp_widths=xh.ANALYSIS_WIDTHS['reco_costheta']),
}
TRUE_LABELS = {vn: vc.xlabel_true for vn, vc in VAR_CFGS.items()}
# dσ/dx units: KE bins are in MeV, the cross section is quoted per GeV
X_UNIT = {'reco_ke': 1e-3, 'reco_costheta': 1.0}
XSEC_YLABEL = {
    'reco_ke':       r'd$\sigma$/d$E_e$ [cm$^2$ GeV$^{-1}$ nucleon$^{-1}$]',
    'reco_costheta': r'd$\sigma$/d$\cos\theta_e$ [cm$^2$ nucleon$^{-1}$]',
}
for vn, vc in VAR_CFGS.items():
    print(f'{vn:<14s} {vc.n_bins} bins: {vc.bins}')

## 1. Load selection
`is_sig` (true $\nu_e$ CC in FV, $T_e>500$ MeV) defines signal everywhere in this notebook.
The efficiency denominator is the true-signal table `nuecc_dfs/true_signal_nuecc.df` (one row per true signal $\nu$,
including $\nu$ lost at flash matching / FV / reconstruction), written by `nue_selection.ipynb`. The saved
signal covariances (`cov_ms_ms`) were built from `truth_cat == 0`, so the two definitions are compared here.

In [ ]:
st = nh.load_sel_topo(SEL_FILE, df_file=None, verbose=False)
sel_topo, pot_scale = st['sel_topo'], st['pot_scale']
true_sig = st['true_sig']     # one row per true signal ν incl. ν lost before presel (efficiency denominator)

is_sig = sel_topo['is_sig'].astype(bool)
fin    = sel_topo[FINAL_STAGE].astype(bool)
cat0   = sel_topo['truth_cat'] == 0
print(f'sel_topo: {len(sel_topo):,} interactions   pot_scale (MC → {TARGET_POT:.1e}) = {pot_scale:.4f}')
n_true = len(true_sig) if true_sig is not None else int(is_sig.sum())
n_nu_sel = int(true_sig[FINAL_STAGE].sum()) if true_sig is not None else int((fin & is_sig).sum())
print(f'True signal ν (pre-cut):         {n_true:>7,}  → {n_true*pot_scale:9.1f} at {TARGET_POT:.1e} POT')
print(f'  preselected signal interactions: {is_sig.sum():>7,}')
print(f'Selected at {FINAL_STAGE}: {fin.sum():>7,}  (signal {(fin & is_sig).sum():,} interactions from {n_nu_sel:,} ν)')
n_mis = int((fin & (is_sig != cat0)).sum())
print(f'Selected events where is_sig ≠ (truth_cat==0): {n_mis}'
      + ('' if n_mis == 0 else '  ⚠ signal CV here and cov_ms_ms in nue_syst use different definitions'))
print(f'Efficiency = {n_nu_sel/n_true:.3f}   '
      f'Purity (MC only: ν + MC cosmic) = {(fin & is_sig).sum()/fin.sum():.3f}   '
      '(with off-beam: §3)')
print('Selected by truth_cat: ' + ', '.join(f'{int(c)}: {n:,}' for c, n in
                                         sel_topo.loc[fin, 'truth_cat'].value_counts().sort_index().items()))

## 2. Response matrix, migration, efficiency  (technote Eq. 32, Fig. 33)
$r_{a\mu} = \epsilon_\mu\,\Delta_{a\mu}$: $\Delta$ is the migration (columns sum to 1), $\epsilon$ the efficiency,
so columns of $r$ sum to $\epsilon_\mu$. True bins = reco bins (square), under/overflow in the first/last bin.
Plotted in the same format as the former `nue_syst.ipynb` §6.5 (rows = true, columns = reco, designed display widths);
the migration matrix is the row-normalized matrix that §6.5 called "response".
The model $\phi$ (all true signal) is POT-scaled, so $r\cdot\phi$ reproduces the selected-signal reco CV exactly.

In [ ]:
resp = {}
for vn, vc in VAR_CFGS.items():
    r = xh.build_response(sel_topo, FINAL_STAGE, vn, xh.TRUE_VAR[vn], vc.bins, pot_scale,
                          truth_df=true_sig)
    resp[vn] = r
    closure = np.max(np.abs(r['response'] @ r['model'] - r['sig_reco']))
    print(f"\n{vn}:  true signal {r['n_sig_true']:,}  selected {r['n_sig_sel']:,}  "
          f"(|r·φ − S_reco|max = {closure:.2e})")
    print(f"  efficiency       = {np.round(r['efficiency'], 3)}")
    print(f"  migration diag   = {np.round(np.diag(r['migration']), 3)}")
    print(f"  response diag    = {np.round(np.diag(r['response']), 3)}")

In [ ]:
for vn, vc in VAR_CFGS.items():
    r = resp[vn]
    # arrays are (reco a, true μ) → transpose so rows = true, columns = reco
    for M, name, title, fmt in [
            (r['migration'].T, 'migration_matrix', f'Migration matrix P(reco | true, selected): {vn}', '{:.2f}'),
            (r['response'].T,  'response_matrix',  rf'Response matrix $r=\epsilon\,\Delta$: {vn}',   '{:.3f}')]:
        fig, ax = xh.plot_designed_matrix(M, vc, TRUE_LABELS[vn], title, fmt=fmt)
        savefig(fig, f'{name}_{vn}', 'response')

    fig, ax = plt.subplots(figsize=(7, 4.5))
    eff = r['efficiency']
    err = np.sqrt(np.clip(eff * (1 - eff), 0, None) / np.clip(r['truth_raw'], 1, None))
    step(ax, vc.disp_edges, eff * 100, color='steelblue', lw=2)
    ax.errorbar(vc.disp_centers, eff * 100, yerr=err * 100, fmt='none', ecolor='steelblue', capsize=3)
    tot = r['n_sig_sel'] / r['n_sig_true']
    ax.axhline(tot * 100, color='gray', ls='--', lw=1, label=f'Integrated: {tot*100:.1f}%')
    vc.set_xaxis(ax, TRUE_LABELS[vn]); ax.set_ylim(0, 100)
    ax.set_ylabel('Selection efficiency [%]'); ax.legend(frameon=False); ax.grid(alpha=0.2, ls='--')
    ax.set_title(f'{TITLE}: selection efficiency', fontsize=12)
    fig.tight_layout()
    savefig(fig, f'efficiency_{vn}', 'response')

## 3. Background, covariances, expected event rate
* **Background** $B$ = selected MC that is not signal (all `truth_cat`) + dirt + off-beam (from `bkg_extra_cv.npz`).
* **Covariance of the prediction** per family = $V_{SS}+V_{BB}+V_{SB}+V_{BS}$ (technote: $V_{m_S m_S}$), summed over
  all families in `saved_syst/<var>/`. Files whose shape does not match the binning (e.g. the old 7-bin
  `detsys_cov_matrices.npz` from Aug.) are skipped and reported.
* **Data statistics**: Poisson, $\mathrm{diag}(n_\mathrm{pred})$ at `EXPOSURE_POT`.

In [ ]:
SCALE = EXPOSURE_POT / TARGET_POT
ana = {}
for vn, vc in VAR_CFGS.items():
    r = resp[vn]
    # ── background CVs ──────────────────────────────────────────────────────────
    cats = xh.mc_reco_by_category(sel_topo, FINAL_STAGE, vn, vc.bins, pot_scale)
    f_extra = os.path.join(SYST_DIR, vn, 'bkg_extra_cv.npz')
    if os.path.exists(f_extra):
        d = np.load(f_extra)
        assert np.allclose(d['bins'], vc.bins), f'{f_extra}: bins {d["bins"]} ≠ {vc.bins}'
        extra = {'dirt': d['dirt_cv'].astype(float), 'offbeam': d['offbeam_cv'].astype(float)}
    else:
        warnings.warn(f'{f_extra} missing → dirt/off-beam NOT in the background. '
                      'Run nue_syst.ipynb through §2.11b to create it.')
        extra = {'dirt': np.zeros(vc.n_bins), 'offbeam': np.zeros(vc.n_bins)}

    # ── covariances (6.6e20) ────────────────────────────────────────────────────
    covs, ref, skipped = xh.load_family_covs(SYST_DIR, vn, vc.n_bins)
    rel_ms = np.max(np.abs(ref['ms'] - r['sig_reco'])) / max(ref['ms'].max(), 1)
    mc_bkg = sum(cats.get(c, 0) for c in range(1, 8))
    rel_bs = np.max(np.abs(ref['bs'] - mc_bkg)) / max(ref['bs'].max(), 1)
    print(f'[{vn}] NPZ vs notebook CV: signal max rel. diff {rel_ms:.2e}, bkg(cat1-7) {rel_bs:.2e}')
    if max(rel_ms, rel_bs) > 0.01:
        warnings.warn(f'[{vn}] saved CVs differ from this notebook by >1% → binning/selection changed; '
                      're-run nue_syst.ipynb')

    # ── scale everything to EXPOSURE_POT ────────────────────────────────────────
    sig = r['sig_reco'] * SCALE
    comps = {'signal': sig, **{c: h * SCALE for c, h in cats.items() if c != 'signal'},
             'dirt': extra['dirt'] * SCALE, 'offbeam': extra['offbeam'] * SCALE}
    bkg = sum(v for k, v in comps.items() if k != 'signal')
    V_fam = {fam: xh.full_pred_cov(b) * SCALE**2 for fam, b in covs.items()}
    V_sig = {fam: b['cov_ms_ms'] * SCALE**2 for fam, b in covs.items()}
    n_pred = sig + bkg
    ana[vn] = dict(model=r['model'] * SCALE, sig=sig, bkg=bkg, comps=comps, n_pred=n_pred,
                   V_fam=V_fam, V_sig=V_sig, V_syst=sum(V_fam.values()),
                   V_stat=np.diag(n_pred), skipped=skipped)
    ana[vn]['purity'] = sig.sum() / n_pred.sum()                   # signal / everything predicted
    _get = lambda k: comps.get(k, np.zeros(vc.n_bins)).sum()
    print(f'[{vn}] at {EXPOSURE_POT:.2e} POT: S={sig.sum():.1f}  B={bkg.sum():.1f} '
          f'(other ν {_get(6):.1f}, MC cosmic {_get(7):.1f}, dirt ν {_get("dirt"):.1f}, '
          f'off-beam {_get("offbeam"):.1f})  purity = {ana[vn]["purity"]:.3f}')

### 3.1 Expected event rate in the signal region (technote Fig. 5 style)

In [ ]:
MC_CATS = [0, 1, 2, 3, 4, 5, 6, 7]  # nue_plotter.plot_handscan_binning order: signal at the bottom

def stack_style(k):
    if k == 'dirt':    return nh.CAT_LABELS[9], nh.CAT_COLORS[9]
    if k == 'offbeam': return nh.CAT_LABELS[8], nh.CAT_COLORS[8]
    return nh.CAT_LABELS[k], nh.CAT_COLORS[k]

def draw_stack(ax, vc, comps, band_cov=None, data=None, data_label='Asimov data', pot_text=POT_LABEL):
    """Same stacking / legend as plot_handscan_binning: truth_cat 0 → 7 from the bottom, then
    dirt, off-beam on top. Legend: (events, % of the full prediction) — the signal % is the purity."""
    e, bw, n = vc.disp_edges, np.diff(vc.disp_edges), vc.n_bins
    get = lambda k: comps.get('signal' if k == 0 else k, np.zeros(n))
    mc    = [(k, get(k)) for k in MC_CATS]
    extra = [(k, get(k)) for k in ('dirt', 'offbeam')]
    mc_tot = sum(h.sum() for _, h in mc)
    total  = sum(h for _, h in mc + extra)
    grand  = total.sum()
    bottom, handles, labels = np.zeros(n), [], []
    for k, h in mc + extra:
        if h.sum() <= 0:
            continue
        lbl, col = stack_style(k)
        ax.bar(e[:-1], h, width=bw, bottom=bottom, align='edge', color=col, edgecolor='none', linewidth=0)
        bottom = bottom + h
        pct = h.sum() / grand                       # fraction of the full prediction: signal % = purity
        handles.append(mpatches.Patch(facecolor=col, edgecolor='none'))
        labels.append(f'{lbl} ({h.sum():.1f}, {pct:.1%})')
    top = total.max()
    if data is not None:
        ax.errorbar(vc.disp_centers, data, yerr=np.sqrt(np.clip(data, 1, None)), fmt='ko', markersize=4, zorder=10)
        handles.append(Line2D([], [], color='black', marker='o', linestyle='None', markersize=4))
        labels.append(data_label)
        top = max(top, (data + np.sqrt(np.clip(data, 1, None))).max())
    if band_cov is not None:
        unc = np.sqrt(np.clip(np.diag(band_cov), 0, None))
        ax.bar(e[:-1], 2 * unc, bottom=total - unc, width=bw, align='edge', alpha=0.25, color='gray',
               hatch='///', linewidth=0, zorder=5)
        handles.append(mpatches.Patch(facecolor='gray', edgecolor='black', alpha=0.25, hatch='///', linewidth=0))
        labels.append('Syst. unc.')
    mid = n // 2
    side = 'right' if total[mid:].max() <= total[:mid].max() else 'left'
    ax.legend(handles, labels, fontsize=7, ncol=2, loc=f'upper {side}', frameon=True,
              framealpha=0.85, edgecolor='none', title=f'{pot_text} — total MC events: {grand:.0f}',
              title_fontsize=7)
    ax.set_ylim(0, top * 1.6)
    ax.set_ylabel('Events / bin', fontsize=12)
    return total

for vn, vc in VAR_CFGS.items():
    a = ana[vn]
    fig, (ax, ar) = plt.subplots(2, 1, figsize=(7.0, 7.5), sharex=True,
                                 gridspec_kw={'height_ratios': [3.2, 1]})
    plt.subplots_adjust(hspace=0.05)
    draw_stack(ax, vc, a['comps'], band_cov=a['V_syst'], data=a['n_pred'])
    ax.set_title(f'{TITLE} — signal region (Asimov)', fontsize=12)
    fs = np.sqrt(np.diag(a['V_syst'])) / a['n_pred']
    ar.bar(vc.disp_edges[:-1], 2 * fs, width=np.diff(vc.disp_edges), bottom=1 - fs, align='edge',
           alpha=0.25, color='gray', hatch='///', linewidth=0)
    ar.errorbar(vc.disp_centers, np.ones(vc.n_bins), yerr=1 / np.sqrt(a['n_pred']), fmt='ko', ms=3)
    ar.axhline(1, color='gray', ls='--', lw=1); ar.set_ylim(0.5, 1.5); ar.set_ylabel('Data/MC', fontsize=11)
    vc.set_xaxis(ar, vc.xlabel)
    savefig(fig, f'{vn}_event_rate_asimov', 'uncertainty')

### 3.2 Uncertainty budget on the cross section (technote Figs. 19 & 22)
**Event rate vs. cross section.** `nue_syst.ipynb` (§2.11 table, §6) quotes uncertainties on the *event rate*:
$\sqrt{\sum V}/\sum(S+B)$. Those stay there. The extra cross-section dials (MINERvA 2p2h, MiscInteraction,
NOvA non-resonant π — formerly "Other xsec") are part of the GENIE family, as in the technote.
The cross section is proportional to the background-subtracted signal $D-B$, so here the same covariance is
divided by $S$ only: per bin $\sqrt{V_{aa}}/S_a$, normalization $\sqrt{\sum_{ab}V_{ab}}/\sum_a S_a$ (technote legend
convention). Because $B$ is removed but its uncertainty is not, the xsec numbers are larger than the event-rate ones.
Caveat: the signal part of GENIE/flux is still the *rate* variation (see §8), so these are conservative.

In [ ]:
FAM_LABEL = {'genie': 'GENIE', 'flux': 'Flux', 'extra_xsec': 'Extra xsec', 'g4': 'G4',
             'detsys': 'Detector variations', 'pot': 'Beam exposure', 'ntargets': r'$N_\mathrm{targets}$',
             'cosmic': 'Cosmic', 'mcstat': 'MC statistics'}
FAM_COLOR = {'genie': '#0072B2', 'flux': '#E69F00', 'extra_xsec': '#56B4E9', 'g4': '#009E73',
             'detsys': '#CC79A7', 'pot': '#999999', 'ntargets': '#7F7F7F', 'cosmic': '#F0E442',
             'mcstat': '#D55E00'}
# technote normalization % (Fig. 19, reco E_e, 8.81e19 POT) for orientation
TECHNOTE_NORM = {'genie': 4.1, 'flux': 8.4, 'detsys': 4.4, 'g4': 1.5, 'pot': 2.5, 'ntargets': 1.2,
                 'cosmic': 0.5, 'mcstat': 1.9, 'data stat.': 4.0, 'total': 11.7}

budget = {}
for vn, vc in VAR_CFGS.items():
    a = ana[vn]; S = a['sig']
    fam_order = sorted(a['V_fam'], key=lambda f: -xh.norm_frac(a['V_fam'][f], S))
    rows = {}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for fam in fam_order:
        V = a['V_fam'][fam]
        fr = np.sqrt(np.clip(np.diag(V), 0, None)) / S
        nf = xh.norm_frac(V, S)
        rows[fam] = nf * 100
        step(ax1, vc.disp_edges, fr * 100, color=FAM_COLOR.get(fam), lw=1.6,
             label=f'{FAM_LABEL.get(fam, fam)} ({nf*100:.1f}%)')
    V_tot = a['V_syst'] + a['V_stat']
    STAT = (a['V_stat'], 'Data statistics', dict(color='k', ls=':', lw=2))
    TOT  = (V_tot, 'Total', dict(color='k', lw=2.4))
    SYST = (a['V_syst'], 'Total syst.', dict(color='#0072B2', lw=2))
    for ax, curves in [(ax1, [STAT, TOT]), (ax2, [SYST, STAT, TOT])]:
        for V, lab, kw in curves:
            step(ax, vc.disp_edges, np.sqrt(np.diag(V)) / S * 100,
                 label=f'{lab} ({xh.norm_frac(V, S)*100:.1f}%)', **kw)
    rows['data stat.'] = xh.norm_frac(a['V_stat'], S) * 100
    rows['total syst.'] = xh.norm_frac(a['V_syst'], S) * 100
    rows['total'] = xh.norm_frac(V_tot, S) * 100
    budget[vn] = rows
    for ax, ttl in [(ax1, 'Breakdown by source'), (ax2, 'Systematic vs. statistical')]:
        vc.set_xaxis(ax, vc.xlabel); ax.set_ylim(0, None); ax.grid(alpha=0.2, ls='--')
        ax.set_ylabel('Uncertainty on the cross section [%]')
        ax.set_title(f'{ttl} — {POT_LABEL}', fontsize=11)
        ax.legend(fontsize=8, frameon=False, title='Normalization %', title_fontsize=8)
    ax1.set_ylim(0, ax1.get_ylim()[1] * 1.15)
    fig.tight_layout()
    savefig(fig, f'{vn}_xsec_uncertainty_budget', 'uncertainty')

tab = pd.DataFrame(budget)
tab['technote (E_e)'] = pd.Series(TECHNOTE_NORM)
print(f'Normalization uncertainty on the signal [%]  ({POT_LABEL}; technote: 8.81e19 POT, after its own choices)')
with pd.option_context('display.float_format', '{:.2f}'.format):
    display(tab)

### 3.3 Fractional covariance and correlation of the total prediction uncertainty (technote Figs. 20–21)

In [ ]:
for vn, vc in VAR_CFGS.items():
    a = ana[vn]
    V = a['V_syst'] + a['V_stat']
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    plot_matrix(axes[0], xh.frac_cov(V, a['n_pred']), vc, cmap='viridis', fmt='{:.4f}',
                title='Fractional covariance (syst. + data stat.)', ylabel=vc.xlabel)
    plot_matrix(axes[1], xh.corr_from_cov(V), vc, cmap='RdBu_r', vmin=-1, vmax=1,
                title='Correlation', ylabel=vc.xlabel)
    fig.tight_layout()
    savefig(fig, f'{vn}_total_fraccov_corr', 'uncertainty')

## 4. Wiener-SVD unfolding  (technote §5.2–5.3)
### 4.1 Closure test (Fig. 34)
Asimov signal $d = r\cdot\phi$, covariance = MC statistics only. Expect $\chi^2\approx0$, $p\approx1$.

In [ ]:
def xsec_den(vn):
    return xh.xsec_denominator(VAR_CFGS[vn].bin_widths * X_UNIT[vn], EXPOSURE_POT, N_TARGETS, FLUX_PER_POT)

closure = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, (vn, vc) in zip(axes, VAR_CFGS.items()):
    a, r = ana[vn], resp[vn]
    d = r['response'] @ a['model']
    V_mc = a['V_sig'].get('mcstat', np.diag(np.clip(d, 1, None)))
    V_mc = V_mc + 1e-9 * np.eye(vc.n_bins) * np.trace(V_mc)
    uf = xh.unfold(WienerSVD, r['response'], a['model'], d, V_mc, C_TYPE, NORM_TYPE)
    closure[vn] = uf
    den = xsec_den(vn)
    step(ax, vc.disp_edges, uf['model_smeared'] / den, color='#56B4E9', lw=2, label=r'$A_C\otimes$ true signal')
    ax.errorbar(vc.disp_centers, uf['unfold'] / den, yerr=np.sqrt(np.clip(np.diag(uf['UnfoldCov']), 0, None)) / den,
                fmt='ko', ms=4, capsize=3, label='Unfolded Asimov (MC stat. only)')
    ax.text(0.04, 0.95, rf"$\chi^2$/dof = {uf['chi2']:.2g}/{uf['ndf']}, $p$ = {uf['p']:.2f}",
            transform=ax.transAxes, va='top')
    vc.set_xaxis(ax, TRUE_LABELS[vn]); ax.set_ylabel(XSEC_YLABEL[vn])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.25); ax.legend(frameon=False, loc='upper right', fontsize=8)
    print(f"{vn}: closure χ²/ndf = {uf['chi2']:.3g}/{uf['ndf']}  p = {uf['p']:.3f}   "
          f"max |U − A_C φ|/σ = {np.max(np.abs(uf['unfold']-uf['model_smeared'])/np.sqrt(np.diag(uf['UnfoldCov']))):.2e}")
fig.suptitle('Unfolding closure test', y=1.0)
fig.tight_layout()
savefig(fig, 'closure_test', 'unfolding')

### 4.2 Nominal Asimov unfolding with full uncertainties
$d = n_\mathrm{pred} - B$, covariance $= V_\mathrm{syst} + \mathrm{diag}(n_\mathrm{pred})$. Each component is also pushed
through the same linear unfolding ($\mathrm{Cov}_\mathrm{unf} = R_\mathrm{tot}\,V\,R_\mathrm{tot}^T$, Eq. 53–54) so the
unfolded uncertainty can be broken down by source.

In [ ]:
nominal = {}
for vn, vc in VAR_CFGS.items():
    a, r = ana[vn], resp[vn]
    d = a['n_pred'] - a['bkg']
    cov = a['V_syst'] + a['V_stat']
    uf = xh.unfold(WienerSVD, r['response'], a['model'], d, cov, C_TYPE, NORM_TYPE)
    uf['cov_stat'] = xh.propagate_cov(uf, a['V_stat'])
    uf['cov_syst'] = xh.propagate_cov(uf, a['V_syst'])
    uf['cov_fam']  = {f: xh.propagate_cov(uf, V) for f, V in a['V_fam'].items()}
    nominal[vn] = uf
    print(f"{vn}: χ²(U vs A_C φ) = {uf['chi2']:.2g}/{uf['ndf']}   Wiener filter = {np.round(uf['WF'], 3)}")
    print(f"   unfolded frac. unc. [%] stat {np.round(np.sqrt(np.diag(uf['cov_stat']))/uf['unfold']*100, 1)}"
          f"   syst {np.round(np.sqrt(np.diag(uf['cov_syst']))/uf['unfold']*100, 1)}")

### 4.3 Additional smearing $A_C$, Wiener filter, unfolded covariance (technote Figs. 40–42)

In [ ]:
for vn, vc in VAR_CFGS.items():
    uf = nominal[vn]
    fig, axes = plt.subplots(1, 4, figsize=(24, 4.8), gridspec_kw={'width_ratios': [1, 1, 1, 0.8]})
    plot_matrix(axes[0], uf['AddSmear'], vc, cmap='viridis', title=r'Additional smearing $A_C$',
                xlabel=TRUE_LABELS[vn], ylabel=TRUE_LABELS[vn])
    den = xsec_den(vn)
    plot_matrix(axes[1], uf['UnfoldCov'] / np.outer(den, den), vc, cmap='viridis', fmt='{:.1e}', fontsize=7,
                title=r'Unfolded covariance [cm$^4$]', xlabel=TRUE_LABELS[vn], ylabel=TRUE_LABELS[vn])
    plot_matrix(axes[2], xh.frac_cov(uf['UnfoldCov'], uf['unfold']), vc, cmap='viridis', fmt='{:.4f}',
                title='Unfolded fractional covariance', xlabel=TRUE_LABELS[vn], ylabel=TRUE_LABELS[vn])
    ax = axes[3]
    ax.bar(np.arange(1, len(uf['WF']) + 1), uf['WF'], color='steelblue')
    ax.set_xlabel('Singular-value index'); ax.set_ylabel('Wiener filter factor')
    ax.set_ylim(0, 1.05); ax.set_title(f'Wiener filter (C{C_TYPE})'); ax.grid(alpha=0.2, ls='--')
    fig.tight_layout()
    savefig(fig, f'{vn}_AC_unfoldcov_wiener', 'unfolding')

### 4.4 Unfolded uncertainty breakdown by source

In [ ]:
for vn, vc in VAR_CFGS.items():
    uf = nominal[vn]; U = uf['unfold']
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for fam, C in sorted(uf['cov_fam'].items(), key=lambda kv: -xh.norm_frac(kv[1], U)):
        step(ax, vc.disp_edges, np.sqrt(np.clip(np.diag(C), 0, None)) / U * 100, color=FAM_COLOR.get(fam), lw=1.6,
             label=f'{FAM_LABEL.get(fam, fam)} ({xh.norm_frac(C, U)*100:.1f}%)')
    for C, lab, kw in [(uf['cov_stat'], 'Data statistics', dict(color='k', ls=':', lw=2)),
                       (uf['UnfoldCov'], 'Total', dict(color='k', lw=2.4))]:
        step(ax, vc.disp_edges, np.sqrt(np.diag(C)) / U * 100, label=f'{lab} ({xh.norm_frac(C, U)*100:.1f}%)', **kw)
    vc.set_xaxis(ax, TRUE_LABELS[vn]); ax.set_ylim(0, None); ax.grid(alpha=0.2, ls='--')
    ax.set_ylabel('Unfolded uncertainty [%]')
    ax.set_title(f'Unfolded (regularized) space — {POT_LABEL}', fontsize=11)
    ax.legend(fontsize=8, frameon=False, title='Normalization %', title_fontsize=8)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    fig.tight_layout()
    savefig(fig, f'{vn}_unfolded_uncertainty_budget', 'unfolding')

### 4.5 Regularization choice (technote App. C, Figs. 63–68)
Same Asimov input, assisting matrix $C_0$ (identity), $C_1$ (flat) and $C_2$ (curvature).
A good choice keeps $A_C$ close to diagonal while keeping the unfolded uncertainties reasonable.

In [ ]:
for vn, vc in VAR_CFGS.items():
    a, r = ana[vn], resp[vn]
    d, cov = a['n_pred'] - a['bkg'], a['V_syst'] + a['V_stat']
    den = xsec_den(vn)
    fig, axes = plt.subplots(1, 4, figsize=(23, 4.6))
    for ax, ct in zip(axes[:3], [0, 1, 2]):
        uf = xh.unfold(WienerSVD, r['response'], a['model'], d, cov, ct, NORM_TYPE)
        plot_matrix(ax, uf['AddSmear'], vc, title=rf'$A_C$ with $C_{ct}$  (mean diag {np.mean(np.diag(uf["AddSmear"])):.2f})',
                    xlabel=TRUE_LABELS[vn], ylabel=TRUE_LABELS[vn])
        off = 0.12 * (ct - 1)
        e = np.sqrt(np.diag(uf['UnfoldCov'])) / den
        axes[3].errorbar(vc.disp_centers + off * np.diff(vc.disp_edges), uf['unfold'] / den, yerr=e,
                         fmt='o', ms=4, capsize=2, label=rf'$C_{ct}$: frac. unc. {np.round(e/(uf["unfold"]/den)*100,0).astype(int)}%')
    step(axes[3], vc.disp_edges, a['model'] / den, color='gray', ls='--', label='True signal (unsmeared)')
    vc.set_xaxis(axes[3], TRUE_LABELS[vn]); axes[3].set_ylabel(XSEC_YLABEL[vn])
    axes[3].legend(fontsize=7, frameon=False); axes[3].set_ylim(0, axes[3].get_ylim()[1] * 1.3)
    fig.tight_layout()
    savefig(fig, f'{vn}_regularization_C0_C1_C2', 'unfolding')

## 5. Fake-data tests (unconstrained)  (technote §5.3.2)
Each test re-weights the MC with one set of weights used for **both** the fake data and the alternative
truth. Fake data $D$ = re-weighted selected MC + unchanged dirt/off-beam; measured $d = D - B_\mathrm{CV}$
(nominal background); covariance $V_\mathrm{FDT}+\mathrm{diag}(D)$ where, as in the technote, $V_\mathrm{FDT}$ holds
only MC statistics + cross-section families (`FDT_FAMILIES`; set it to `None` to use every systematic).

* **Event rate**: $\chi^2_\mathrm{rate}$ of $D$ vs the CV prediction (Eq. 56).
* **Unfolded**: $\chi^2$ against $A_C\otimes$ baseline and $A_C\otimes$ modified truth;
  $\Delta\chi^2 = \chi^2_\mathrm{base} - \chi^2_\mathrm{mod}$ measures model separation.
  Expect: signal-dominant tests → large $\Delta\chi^2$; background-dominant ($\pi^0$) → $\Delta\chi^2\approx0$ and
  $\chi^2_\mathrm{base}$ inflated by the un-constrained background (this is what the CCBC is meant to fix).

Interaction-mode tests (QE/RES/DIS, technote App. B) need a GENIE mode column, which `sel_topo` does not have.

In [ ]:
def fdt_list(vn):
    tc = xh.TRUE_VAR[vn]; lo, hi = VAR_CFGS[vn].bins[[0, -1]]
    return [
        (r'$\pi^0$ bkg. ×0.8 (bkg-dominant)',              'pi0_m20',   dict(pi0_scale=0.8)),
        (r'$\pi^0$ bkg. ×1.2 (bkg-dominant)',              'pi0_p20',   dict(pi0_scale=1.2)),
        (r'Signal tilt $\alpha$=+0.3 (signal-dominant)',   'tilt_p03',  dict(tilt=(tc, 0.3, lo, hi))),
        (r'Signal tilt $\alpha$=−0.3 (signal-dominant)',   'tilt_m03',  dict(tilt=(tc, -0.3, lo, hi))),
        (r'Signal $T_e$<1 GeV ×0.8 (signal-dominant)',     'lowE_m20',  dict(low_e=('true_ke', 1000., 0.8))),
        (r'$\pi^0$ ×1.2 + tilt +0.3 (mixed)',              'mixed',     dict(pi0_scale=1.2, tilt=(tc, 0.3, lo, hi))),
    ]

FDT_FAMILIES = ('mcstat', 'genie')   # technote §5.3.2: MC stat. + cross section; None → all

fdt_results = {}
for vn, vc in VAR_CFGS.items():
    a, r = ana[vn], resp[vn]
    V_fdt = sum(V for f, V in a['V_fam'].items() if FDT_FAMILIES is None or f in FDT_FAMILIES)
    r_exp = {**r, 'model': a['model'], 'sig_reco': a['sig']}              # at EXPOSURE_POT
    extra = a['comps']['dirt'] + a['comps']['offbeam']                    # unchanged in fake data
    den = xsec_den(vn)
    for label, tag, kw in fdt_list(vn):
        w = xh.fdt_weights(sel_topo, **kw)
        w_true = None if true_sig is None else xh.fdt_weights(true_sig, **kw)
        res = xh.run_fdt(sel_topo, FINAL_STAGE, vn, xh.TRUE_VAR[vn], vc.bins, pot_scale * SCALE, w,
                         r_exp, a['bkg'], extra, V_fdt, WienerSVD, C_TYPE, NORM_TYPE,
                         truth_df=true_sig, truth_w=w_true)
        fdt_results[(vn, tag)] = dict(res, label=label)

        fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 5))
        draw_stack(ax0, vc, a['comps'], band_cov=V_fdt, data=res['D_fake'], data_label='Fake data')
        ax0.set_title(rf"Event rate: $\chi^2$/dof = {res['c2_rate']:.1f}/{res['ndf']}, p = {res['p_rate']:.3f}", fontsize=10)
        vc.set_xaxis(ax0, vc.xlabel)

        U, Uc = res['unfold']['unfold'], res['unfold']['UnfoldCov']
        ax1.errorbar(vc.disp_centers, U / den, yerr=np.sqrt(np.clip(np.diag(Uc), 0, None)) / den,
                     fmt='ko', ms=4, capsize=3, label='Unfolded fake data')
        step(ax1, vc.disp_edges, res['ms_nom'] / den, color='#0072B2', ls='--', lw=1.6,
             label=rf"$A_C\otimes$ baseline: $\chi^2$={res['c2_nom']:.1f}/{res['ndf']}, p={res['p_nom']:.2f}")
        step(ax1, vc.disp_edges, res['ms_alt'] / den, color='#D55E00', lw=2,
             label=rf"$A_C\otimes$ modified: $\chi^2$={res['c2_mod']:.1f}/{res['ndf']}, p={res['p_mod']:.2f}")
        vc.set_xaxis(ax1, TRUE_LABELS[vn]); ax1.set_ylabel(XSEC_YLABEL[vn])
        ax1.set_ylim(0, ax1.get_ylim()[1] * 1.3); ax1.legend(fontsize=8, frameon=False, loc='upper right')
        ax1.set_title(rf"Unfolded: $\Delta\chi^2$ = {res['c2_nom'] - res['c2_mod']:.2f}", fontsize=10)
        fig.suptitle(f'FDT — {label} — {vn}', y=1.0)
        fig.tight_layout()
        savefig(fig, f'{vn}_{tag}', 'fakedata')

In [ ]:
rows = []
for (vn, tag), res in fdt_results.items():
    rows.append(dict(variable=vn, test=res['label'].replace('$', ''),
                     chi2_rate=res['c2_rate'], p_rate=res['p_rate'],
                     chi2_base=res['c2_nom'], p_base=res['p_nom'],
                     chi2_mod=res['c2_mod'], p_mod=res['p_mod'],
                     dchi2=res['c2_nom'] - res['c2_mod'], ndf=res['ndf']))
FDT_TABLE = pd.DataFrame(rows).set_index(['variable', 'test'])
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.width', 200):
    display(FDT_TABLE)
print('Pass criterion (technote): p_mod > 0.05 for every test.  '
      f"Failing: {list(FDT_TABLE.index[FDT_TABLE['p_mod'] < 0.05]) or 'none'}")

## 6. Cross section
### 6.1 Flux-averaged total cross section (Eq. 29, Table 7)
$\sigma = \frac{\sum_a (D_a - B_a)}{\epsilon\,N_\mathrm{target}\,\Phi}$ with the integrated efficiency $\epsilon$.
Both variables use the same events, so they must give the same number.

In [ ]:
total_xsec = {}
for vn in VAR_CFGS:
    a = ana[vn]
    eps_tot = a['sig'].sum() / a['model'].sum()
    den = eps_tot * N_TARGETS * PHI
    d = a['n_pred'] - a['bkg']
    sig_tot = d.sum() / den
    stat = np.sqrt(a['V_stat'].sum()) / den
    syst = np.sqrt(max(a['V_syst'].sum(), 0)) / den
    by_fam = {f: np.sqrt(max(V.sum(), 0)) / den for f, V in a['V_fam'].items()}
    total_xsec[vn] = dict(sigma=sig_tot, stat=stat, syst=syst, eff=eps_tot, by_fam=by_fam,
                          genie=a['model'].sum() / (N_TARGETS * PHI))
    print(f"[{vn}]  ε = {eps_tot:.3f}   σ = ({sig_tot/1e-39:.3f} ± {stat/1e-39:.3f} (stat) ± {syst/1e-39:.3f} (syst)) "
          f"× 1e-39 cm²/nucleon   [GENIE CV: {total_xsec[vn]['genie']/1e-39:.3f}]")
    print('    syst by family [1e-39]: ' + ', '.join(f'{FAM_LABEL.get(f, f)} {v/1e-39:.3f}'
                                                 for f, v in sorted(by_fam.items(), key=lambda kv: -kv[1])))
print('\nTechnote (data, 8.81e19 POT): σ = (3.87 ± 0.18 (stat) ± 0.40 (syst)) × 1e-39;  GENIE AR23 4.49e-39')

fig, ax = plt.subplots(figsize=(6.5, 3.2))
t = total_xsec['reco_ke']
ax.errorbar([t['sigma'] / 1e-39], [1], xerr=[np.hypot(t['stat'], t['syst']) / 1e-39], fmt='ko', capsize=4,
            label='Asimov (stat ⊕ syst)')
ax.errorbar([t['sigma'] / 1e-39], [1], xerr=[t['stat'] / 1e-39], fmt='none', ecolor='k', capsize=7, lw=2.5)
ax.plot([t['genie'] / 1e-39], [0], 'D', color='#0072B2', label='GENIE CV (this MC)')
ax.plot([4.49], [-1], 's', color='gray', label='Technote: GENIE AR23 4.49')
ax.errorbar([3.87], [-2], xerr=[np.hypot(0.18, 0.40)], fmt='^', color='gray', capsize=4, label='Technote: data 3.87')
ax.set_yticks([]); ax.set_xlabel(r'$\sigma$ [$10^{-39}$ cm$^2$/nucleon]')
ax.set_ylim(-3, 2.2); ax.legend(fontsize=8, frameon=False, loc='upper left', ncol=2)
ax.set_title(f'Flux-averaged total cross section — {POT_LABEL}', fontsize=10)
fig.tight_layout()
savefig(fig, 'total_xsec', 'xsec')

### 6.2 Flux-averaged differential cross sections (technote Fig. 39 style)
Inner error bar: data statistics; outer: statistics ⊕ shape-only systematics; grey band at the bottom:
normalization systematics (norm/shape decomposition of the unfolded syst. covariance). Predictions are smeared
with $A_C$; the unsmeared truth is shown dashed for reference.

In [ ]:
for vn, vc in VAR_CFGS.items():
    a, uf = ana[vn], nominal[vn]
    den = xsec_den(vn)
    U = uf['unfold'] / den
    C_stat = uf['cov_stat'] / np.outer(den, den)
    C_syst = uf['cov_syst'] / np.outer(den, den)
    C_norm, C_shape, C_mix = xh.matrix_decomp(U, C_syst)
    e_stat  = np.sqrt(np.clip(np.diag(C_stat), 0, None))
    e_shape = np.sqrt(np.clip(np.diag(C_shape), 0, None))
    e_norm  = np.sqrt(np.clip(np.diag(C_norm), 0, None))
    e_outer = np.hypot(e_stat, e_shape)
    pred_s = uf['model_smeared'] / den
    c2, ndf, pv = xh.chi2_cov(uf['unfold'], uf['model_smeared'], uf['UnfoldCov'])

    fig, (ax, ar) = plt.subplots(2, 1, figsize=(7.5, 7), sharex=True,
                                 gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.05})
    e = vc.disp_edges
    ax.bar(e[:-1], e_norm, width=np.diff(e), align='edge', color='gray', alpha=0.45, label='Norm. syst.')
    step(ax, e, pred_s, color='#0072B2', lw=2, label=rf'GENIE CV $\otimes A_C$  ($\chi^2$/dof={c2:.1f}/{ndf}, p={pv:.2f})')
    step(ax, e, a['model'] / den, color='#0072B2', lw=1, ls='--', alpha=0.7, label='GENIE CV (unsmeared)')
    ax.errorbar(vc.disp_centers, U, yerr=e_outer, fmt='none', ecolor='k', capsize=4, lw=1)
    ax.errorbar(vc.disp_centers, U, yerr=e_stat, fmt='ko', ms=5, capsize=0, lw=2.5,
                label='Asimov: stat. (inner), stat.⊕shape syst. (outer)')
    ax.set_ylabel(XSEC_YLABEL[vn]); ax.set_ylim(0, max(U + e_outer) * 1.45)
    ax.legend(fontsize=8, frameon=False, loc='upper left')
    ax.set_title(f'{TITLE} — {POT_LABEL}', fontsize=11)
    ax.ticklabel_format(axis='y', style='sci', scilimits=(-2, 2))
    with np.errstate(divide='ignore', invalid='ignore'):
        ar.bar(e[:-1], 2 * e_norm / pred_s, width=np.diff(e), bottom=1 - e_norm / pred_s, align='edge',
               color='gray', alpha=0.45)
        ar.errorbar(vc.disp_centers, U / pred_s, yerr=e_outer / pred_s, fmt='ko', ms=4, capsize=3)
    ar.axhline(1, color='#0072B2', lw=1); ar.set_ylim(0.5, 1.5); ar.set_ylabel(r'Data / $A_C\otimes$Pred')
    vc.set_xaxis(ar, TRUE_LABELS[vn])
    savefig(fig, f'{vn}_differential_xsec_asimov', 'xsec')
    print(f'{vn}: frac. unc. [%]  stat {np.round(e_stat/U*100,1)}  shape {np.round(e_shape/U*100,1)}  '
          f'norm {np.round(e_norm/U*100,1)}')

In [ ]:
# ── Save the unfolding products ─────────────────────────────────────────────────
for vn, vc in VAR_CFGS.items():
    a, uf, den = ana[vn], nominal[vn], xsec_den(vn)
    xh.save_cov_npz(
        os.path.join(COV_OUT_DIR, f'xsec_asimov_{vn}.npz'),
        bins=vc.bins, exposure_pot=EXPOSURE_POT, n_targets=N_TARGETS, flux_per_pot=FLUX_PER_POT,
        response=resp[vn]['response'], efficiency=resp[vn]['efficiency'], model=a['model'],
        measured=a['n_pred'] - a['bkg'], cov_measured=a['V_syst'] + a['V_stat'],
        unfold=uf['unfold'], unfold_cov=uf['UnfoldCov'], unfold_cov_stat=uf['cov_stat'],
        unfold_cov_syst=uf['cov_syst'], add_smear=uf['AddSmear'], wiener_filter=uf['WF'],
        xsec=uf['unfold'] / den, xsec_cov=uf['UnfoldCov'] / np.outer(den, den),
        xsec_pred_smeared=uf['model_smeared'] / den)

## 7. Wiener-SVD vs OmniFold
**What is compared.**
* *OmniFold* (`Omnifold_SBND/sbnd/covariance/covariance_<src>_true_*.npz`): covariance of the OmniFold-unfolded
  **selected** signal (no efficiency correction), from re-running/reweighting the unfolding in each universe of
  flux, GENIE, extra-xsec, G4, MC stat. plus the ML (training) uncertainty. No background, no data statistics.
* *Wiener-SVD*: the **signal block** ($V_{SS}$) of the same families from `nue_syst`, propagated through the nominal
  unfolding of §4.2 (true-space, efficiency-corrected, $A_C$-regularized).

Only fractions and correlations are comparable (OmniFold uses true $p$ and the technote $\cos\theta$ edges).
**Full** covariances are dominated by one normalization mode (correlations ≈ 0.8–1, effective DoF ≈ 1), which hides
the bin-to-bin information, so every plot is repeated **shape-only** (normalization component removed with the
norm/shape decomposition, $V_\mathrm{shape} = V - \frac{N N^T}{N_T^2}\sum V - \dots$).

In [ ]:
OF_VAR  = {'reco_ke': 'true_p', 'reco_costheta': 'true_costheta'}
OF_FAMS = ('flux', 'genie', 'g4', 'mcstat')
OF_XLABEL = {'true_p': r'True leading-$e^-$ momentum [MeV/c]', 'true_costheta': r'True leading-$e^-$ $\cos\theta$'}

def _load_of(src, ov):
    f = os.path.join(OMNIFOLD_COV, f'covariance_{src}_{ov}.npz')
    return np.load(f, allow_pickle=True) if os.path.exists(f) else None

def _shape(cov, mean):
    return xh.matrix_decomp(mean, cov)[1]          # pure shape term (positive semi-definite)

of_cmp = {}
for vn, vc in VAR_CFGS.items():
    ov = OF_VAR[vn]
    of = _load_of('all', ov)
    if of is None or of['cov'].shape[0] != vc.n_bins:
        print(f'{vn}: OmniFold covariance missing or different #bins — skipped'); continue
    of_cov, of_mean = of['cov'], of['mean_hist']
    of_cfg = VariableConfig(vn, of['bins'], OF_XLABEL[ov])
    if not np.allclose(of['bins'], vc.bins):
        print(f'⚠ {vn}: OmniFold bins {of["bins"]} vs W-SVD bins {vc.bins} — compare fractions/correlations only')
    uf = nominal[vn]
    w_mean = uf['unfold']
    w_cov  = xh.propagate_cov(uf, sum(ana[vn]['V_sig'][f] for f in OF_FAMS if f in ana[vn]['V_sig']))
    m = xh.compute_comparison_metrics(w_cov, of_cov, w_mean, of_mean)
    xh.print_comparison_metrics(m, vn)
    w_sh, of_sh = _shape(w_cov, w_mean), _shape(of_cov, of_mean)
    of_cmp[vn] = dict(w_cov=w_cov, of_cov=of_cov, w_mean=w_mean, of_mean=of_mean, w_sh=w_sh, of_sh=of_sh)

    # ── per-family normalization: where does the difference come from? ───────────
    rows = {}
    for fam in OF_FAMS + ('ml',):
        o = _load_of(fam, ov)
        r_ = {'OmniFold norm %': xh.norm_frac(o['cov'], o['mean_hist']) * 100 if o is not None else np.nan}
        if fam in ana[vn]['V_sig']:
            Vs = ana[vn]['V_sig'][fam]
            r_['syst signal (reco) norm %'] = xh.norm_frac(Vs, ana[vn]['sig']) * 100
            r_['W-SVD unfolded norm %'] = xh.norm_frac(xh.propagate_cov(uf, Vs), w_mean) * 100
        rows[fam] = r_
    t = pd.DataFrame(rows).T
    t['OF / W-SVD'] = t['OmniFold norm %'] / t['W-SVD unfolded norm %']
    print(f'\n[{vn}] normalization uncertainty by family')
    with pd.option_context('display.float_format', '{:.2f}'.format):
        display(t)

    # ── correlations: full and shape-only ─────────────────────────────────────────
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 9.5))
    for row, (Cw, Co, tag) in enumerate([(w_cov, of_cov, 'full'), (w_sh, of_sh, 'shape-only')]):
        plot_matrix(axes[row, 0], xh.corr_from_cov(Cw), vc, cmap='RdBu_r', vmin=-1, vmax=1,
                    title=f'Wiener-SVD ({tag})', xlabel=TRUE_LABELS[vn], ylabel=TRUE_LABELS[vn])
        plot_matrix(axes[row, 1], xh.corr_from_cov(Co), of_cfg, cmap='RdBu_r', vmin=-1, vmax=1,
                    title=f'OmniFold ({tag})')
    fig.suptitle(f'Correlation in true space — {vn}', y=1.0); fig.tight_layout()
    savefig(fig, f'corr_truth_{vn}', 'omnifold_comparison')

    # ── per-bin fractional uncertainty: full and shape-only ──────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    x = np.arange(vc.n_bins)
    for ax, (Cw, Co, tag) in zip(axes, [(w_cov, of_cov, 'full'), (w_sh, of_sh, 'shape-only')]):
        fw = np.sqrt(np.clip(np.diag(Cw), 0, None)) / np.abs(w_mean) * 100
        fo = np.sqrt(np.clip(np.diag(Co), 0, None)) / np.abs(of_mean) * 100
        nf = (lambda C, m: f' (norm {xh.norm_frac(C, m)*100:.1f}%)') if tag == 'full' else (lambda C, m: '')
        ax.bar(x - 0.18, fw, 0.35, color='steelblue', label='Wiener-SVD' + nf(Cw, w_mean))
        ax.bar(x + 0.18, fo, 0.35, color='crimson',   label='OmniFold' + nf(Co, of_mean))
        ax.set_xticks(x)
        ax.set_xticklabels([f'{a:g}–{b:g}' for a, b in zip(vc.bins[:-1], vc.bins[1:])], fontsize=8, rotation=30)
        ax.set_xlabel(f'{TRUE_LABELS[vn]} (W-SVD bin edges)'); ax.set_ylabel('Frac. unc. [%]')
        ax.set_title(f'Per-bin uncertainty ({tag})', fontsize=11); ax.legend(fontsize=8, frameon=False)
        ax.grid(alpha=0.2, ls='--', axis='y')
    fig.tight_layout()
    savefig(fig, f'frac_unc_{vn}', 'omnifold_comparison')

    # ── eigenvalue spectra: full and shape-only ──────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (Cw, Co, tag) in zip(axes, [(w_cov, of_cov, 'full'), (w_sh, of_sh, 'shape-only')]):
        for C, mean, lab, kw in [(Cw, w_mean, 'Wiener-SVD', dict(color='steelblue', marker='o')),
                                 (Co, of_mean, 'OmniFold', dict(color='crimson', marker='s'))]:
            ev = np.sort(np.clip(np.linalg.eigvalsh(xh.frac_cov(C, mean)), 1e-12, None))[::-1]
            ax.semilogy(np.arange(1, len(ev) + 1), ev, ms=5, lw=1.5, label=lab, **kw)
        ax.set_xlabel('Eigenvalue index'); ax.set_ylabel('Eigenvalue of fractional covariance')
        ax.set_title(f'Eigenspectrum ({tag})', fontsize=11); ax.legend(fontsize=9); ax.grid(alpha=0.2, ls='--')
    fig.tight_layout()
    savefig(fig, f'eigenspectrum_{vn}', 'omnifold_comparison')

## 8. Sanity checks & summary

In [ ]:
print('=' * 72)
print(f'{"Cross-section analysis summary":^72}')
print('=' * 72)
print(f'Exposure (Asimov): {EXPOSURE_POT:.2e} POT   covariances from {SYST_DIR} at {TARGET_POT:.2e} POT')
print(f'N_targets = {N_TARGETS:.4e} nucleons   Φ = {PHI:.3e} cm⁻²   FINAL_STAGE = {FINAL_STAGE}')
for vn, vc in VAR_CFGS.items():
    a, uf = ana[vn], nominal[vn]
    print(f'\n{vn} ({vc.n_bins} bins)')
    print(f'  S = {a["sig"].sum():.1f}  B = {a["bkg"].sum():.1f}  purity = {a["purity"]:.3f}  '
          f'ε = {total_xsec[vn]["eff"]:.3f}')
    print(f'  families used: {sorted(a["V_fam"])}   skipped: {a["skipped"] or "none"}')
    print(f'  closure χ² = {closure[vn]["chi2"]:.2g}/{closure[vn]["ndf"]}   '
          f'norm. unc. on xsec: syst {budget[vn]["total syst."]:.1f}%, stat {budget[vn]["data stat."]:.1f}%, '
          f'total {budget[vn]["total"]:.1f}%')

# Implied GENIE total cross section from the MC truth + constants (technote GENIE AR23: 4.49e-39)
s_mc = total_xsec['reco_ke']['genie']
print(f'\nImplied GENIE σ from this MC = {s_mc:.3e} cm²/nucleon  (technote AR23: 4.49e-39; '
      f'ratio {s_mc/4.49e-39:.2f})')
print("""
Known limitations / next steps
  0b. OmniFold export (nue_syst §7.2): weights_2d = r_KE(u)·r_cosθ(u) applies each universe's shift twice,
     so OmniFold flux/GENIE/G4 uncertainties are ~2× too large (see §7 table).
  1. GENIE signal-normalization: cov_ms_ms is a *rate* covariance, so the GENIE total-xsec variation of the
     signal is counted as an uncertainty. For the xsec it should be the 'xsec' mode (true signal fixed,
     response varied per universe; see analysis_village/unfolding/utils.get_genie_univs).
  2. Detector systematics: saved_syst/<var>/detsys_cov_matrices.npz is from the old 7-bin binning → skipped.
  3. CCBC (technote §4.2): needs signal-region × sideband covariances from nue_syst (sideband universes).
  4. Interaction-mode FDTs: add the GENIE mode / true Eν columns to the selection DataFrame.
  5. Regularization: for cosθ, C2 gives a non-diagonal A_C (see §4.5) — compare C1 / NORM_TYPE=0.5.
  6. Data unfolding: replace n_pred by the on-beam counts (+ off-beam subtraction) once unblinded.
""")

### Key outputs and how to read them
| Output | Where | What it tells you |
|---|---|---|
| Migration / response / efficiency | §2, `plots_xsec/response/` | Migration diagonal ≳ 0.7–0.95 = good resolution for the chosen bins; response = efficiency × migration is what the unfolding inverts. |
| Signal-region event rate (Asimov) | §3.1 | The prediction that gets unfolded: signal on the bottom, every background that is subtracted, syst band. |
| Uncertainty budget on the xsec | §3.2 table + plot | Which systematic dominates the *measurement* (normalization % in the legend); compare with the technote column. |
| Closure test | §4.1 | Must give χ² ≈ 0, p ≈ 1: the response and unfolding code are self-consistent. |
| $A_C$ + Wiener filter | §4.3, §4.5 | How much regularization smears the truth; any model must be multiplied by $A_C$ before comparing. |
| Fake-data table | §5 | Signal-like shifts must be recovered (p_mod > 0.05, Δχ² > 0); π⁰ (background) shifts should give Δχ² ≈ 0. |
| Total σ and dσ/dx | §6 | The measurement format (stat. inner, stat.⊕shape outer, norm. band); for Asimov the points sit on $A_C\otimes$GENIE by construction, so read the **error bars**. |
| W-SVD vs OmniFold | §7 | Same systematics, two unfolding methods; use the shape-only rows to compare bin-to-bin information. |